<a href="https://colab.research.google.com/github/emily-escudero/Analitica-Educacion-rural/blob/main/Entrega2EducacionRural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests

# Define the API URL
api_url = "https://www.datos.gov.co/api/v3/views/ji8i-4anb/query.json"

# Fetch data from the API
response = requests.get(api_url)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
data = response.json()

# The actual data is usually under a 'data' key or similar structure. Inspect the JSON to confirm.
# For many Socrata APIs, the 'data' key contains the rows, and 'columns' contains metadata.
# Let's assume the data is directly in the response. If not, further inspection of 'data' will be needed.

# Convert to DataFrame. The structure of the JSON needs to be understood. If it's a list of lists,
# we might need column names separately. If it's a list of dictionaries, pd.DataFrame will work directly.
# Based on typical Socrata query.json output, it's often a list of lists, with column headers in a 'meta' or 'columns' section.
# Let's assume for now the JSON structure is a list of dictionaries or similar that pandas can parse directly.
# If not, I will refine this in the next step based on the actual JSON structure.

df = pd.DataFrame(data)
display(df.head())

,:id,:version,:created_at,:updated_at,ano,c_digo_departamento,departamento,poblacion_5_16,tasa_matriculacion_5_16,cobertura_neta,...,reprobacion,reprobacion_transicion,reprobacion_primaria,reprobacion_secundaria,reprobacion_media,repitencia,repitencia_transicion,repitencia_primaria,repitencia_secundaria,repitencia_media
0,row-xccu-54yf.9ie2,rv-69kt.ykig.crwc,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,5,Antioquia,1288473,94.01,93.85,...,2.06,0.07,94.56,2.54,2.96,4.25,0.07,4.56,5.27,1.68
1,row-755p~4uyd-2k3w,rv-ar5t~pmwz_z3hy,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,8,Atlántico,523935,99.32,99.05,...,0.54,0.12,96.49,0.67,0.75,1.82,0.12,1.77,2.18,0.88
2,row-rb29.mx9s.a7ax,rv-atze~ze7f~jiue,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,11,"Bogotá, D.C.",1479334,90.7,90.29,...,0,0,94.69,0,0,3.23,0,2.3,5.11,2.57
3,row-gavf~qv4q~dhdy,rv-9idw~dvm2~k7me,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,13,Bolívar,496676,91.57,91.4,...,2.1,0.46,95.48,2.75,3.67,4.43,0.46,4.44,5.37,2.28
4,row-zz5g~58au~qymy,rv-68rp.uscp_a52j,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,15,Boyacá,300501,86.16,86.11,...,2.73,0.17,96.1,4.31,3.26,2.62,0.17,1.9,4.19,1.55


In [3]:
# Define the columns of interest as provided by the user
desercion_columns = [
    'departamento',
    'ano',
    'desercion',
    'desercion_transicion',
    'desercion_primaria',
    'desercion_secundaria',
    'desercion_media'
]

# Select only the specified columns and explicitly create a copy to avoid SettingWithCopyWarning
df_filtered = df[df.columns.intersection(desercion_columns)].copy()

# Convert 'ano' to numeric to filter by year
df_filtered['ano'] = pd.to_numeric(df_filtered['ano'], errors='coerce')

# Filter for years 2021 to 2024
df_deserciones_2021_2024 = df_filtered[
    (df_filtered['ano'] >= 2021) & (df_filtered['ano'] <= 2024)
].copy()

# Display the first few rows of the filtered DataFrame
display(df_deserciones_2021_2024.head())

,ano,departamento,desercion,desercion_transicion,desercion_primaria,desercion_secundaria,desercion_media
330,2021,Antioquia,4.81,3.59,4.27,5.90,4.13
331,2021,Atlántico,1.67,2.15,1.75,1.68,1.03
332,2021,"Bogotá, D,C,",1.29,1.07,1.08,1.35,1.88
333,2021,Bolívar,3.69,4.13,3.33,4.27,3.13
334,2021,Boyacá,2.97,2.75,2.13,3.63,3.69


"anex-pobrezadepartamental.xlsx"

In [4]:
import pandas as pd
import requests

# Define the raw GitHub URL for the Excel file
excel_url = "https://github.com/emily-escudero/Analitica-Educacion-rural/raw/main/anex-pobrezadepartamental.xlsx"

# Define the local path to save the Excel file
excel_file_path = "anex-pobrezadepartamental.xlsx"

# Download the Excel file
print(f"Downloading {excel_url}...")
response = requests.get(excel_url)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

with open(excel_file_path, 'wb') as f:
    f.write(response.content)

print(f"File downloaded to {excel_file_path}")

File downloaded to anex-pobrezadepartamental.xlsx


### Pobreza Monetaria Data Preparation

Now, I'll load the data from the `Pobreza Monetaria Act.Met.` sheet. Based on your description:
*   Departments are in column `A` from row `12` to `35`.
*   Years are in row `11` from column `B` to `F`.

In [5]:
# Load the 'Pobreza Monetaria Act.Met.' sheet
# We need to read the specific range for departments and years.
# We'll read the data from the relevant cells and then process it.

# Read the header (years) first
years_df = pd.read_excel(
    excel_file_path,
    sheet_name='Pobreza Monetaria Act.Met.',
    header=None, # No default header
    skiprows=10, # Skip to row 11 (0-indexed)
    nrows=1,     # Read only one row for years
    usecols='B:F' # Use columns B to F for years
)

# Extract years from the first row of years_df
years = [int(col) for col in years_df.iloc[0].values]

# Read the data for departments and values
# We skip rows until 11 (row 12 in excel) and read 24 rows (35-12+1)
monetaria_df = pd.read_excel(
    excel_file_path,
    sheet_name='Pobreza Monetaria Act.Met.',
    header=None, # No default header
    skiprows=11, # Skip to row 12 (0-indexed)
    nrows=24,    # From row 12 to 35 (35-12+1 = 24 rows)
    usecols='A:F' # Use columns A to F
)

# Assign column names: first column is 'departamento', then the extracted years
monetaria_df.columns = ['departamento'] + years

# Melt the DataFrame to long format
df_monetaria_melted = monetaria_df.melt(
    id_vars=['departamento'],
    var_name='ano',
    value_name='pobreza_monetaria'
)

# Filter for years 2021 to 2024
df_pobreza_monetaria = df_monetaria_melted[
    (df_monetaria_melted['ano'] >= 2021) & (df_monetaria_melted['ano'] <= 2024)
].copy()

# Display the first few rows
print("Pobreza Monetaria (2021-2024):")
display(df_pobreza_monetaria.head())

Pobreza Monetaria (2021-2024):


,departamento,ano,pobreza_monetaria
0,Antioquia,2021,32.8
1,Atlántico,2021,42.1
2,Bogotá D.C.,2021,30.5
3,Bolívar,2021,54.0
4,Boyacá,2021,41.8


### Pobreza Extrema Data Preparation

Next, I'll process the data from the `Pobreza Extrema Act.Met.` sheet. Based on your description:
*   Departments start in cell `A15` and go up to `A38`.
*   Years are in row `14` from column `B` to `F`.

In [6]:
# Load the 'Pobreza Extrema Act.Met.' sheet

# Read the header (years) first
years_extrema_df = pd.read_excel(
    excel_file_path,
    sheet_name='Pobreza Extrema Act.Met.',
    header=None, # No default header
    skiprows=13, # Skip to row 14 (0-indexed)
    nrows=1,     # Read only one row for years
    usecols='B:F' # Use columns B to F for years
)

# Extract years from the first row of years_extrema_df
years_extrema = [int(col) for col in years_extrema_df.iloc[0].values]

# Read the data for departments and values
# We skip rows until 14 (row 15 in excel) and read 24 rows (38-15+1)
extrema_df = pd.read_excel(
    excel_file_path,
    sheet_name='Pobreza Extrema Act.Met.',
    header=None, # No default header
    skiprows=14, # Skip to row 15 (0-indexed)
    nrows=24,    # From row 15 to 38 (38-15+1 = 24 rows)
    usecols='A:F' # Use columns A to F
)

# Assign column names: first column is 'departamento', then the extracted years
extrema_df.columns = ['departamento'] + years_extrema

# Melt the DataFrame to long format
df_extrema_melted = extrema_df.melt(
    id_vars=['departamento'],
    var_name='ano',
    value_name='pobreza_extrema'
)

# Filter for years 2021 to 2024
df_pobreza_extrema = df_extrema_melted[
    (df_extrema_melted['ano'] >= 2021) & (df_extrema_melted['ano'] <= 2024)
].copy()

# Display the first few rows
print("Pobreza Extrema (2021-2024):")
display(df_pobreza_extrema.head())

Pobreza Extrema (2021-2024):


,departamento,ano,pobreza_extrema
0,Antioquia,2021,9.2
1,Atlántico,2021,12.3
2,Bogotá D.C.,2021,8.4
3,Bolívar,2021,18.8
4,Boyacá,2021,16.4


### Merging DataFrames

Now, I'll merge the two DataFrames (`df_pobreza_monetaria` and `df_pobreza_extrema`) into a single DataFrame based on `departamento` and `ano`.

In [7]:
# Merge the two DataFrames
df_pobreza_merged = pd.merge(
    df_pobreza_monetaria,
    df_pobreza_extrema,
    on=['departamento', 'ano'],
    how='inner'
)

# Display the final DataFrame
print("Final DataFrame with Pobreza Monetaria and Pobreza Extrema (2021-2024):")
display(df_pobreza_merged)

Final DataFrame with Pobreza Monetaria and Pobreza Extrema (2021-2024):


,departamento,ano,pobreza_monetaria,pobreza_extrema
0,Antioquia,2021,32.8,9.2
1,Atlántico,2021,42.1,12.3
2,Bogotá D.C.,2021,30.5,8.4
3,Bolívar,2021,54.0,18.8
4,Boyacá,2021,41.8,16.4
...,...,...,...,...
91,Risaralda,2024,23.8,5.2
92,Santander,2024,27.4,7.7
93,Sucre,2024,57.5,23.7
94,Tolima,2024,35.6,12.2


"anex-educacionformal.xlsx"

In [8]:
import pandas as pd
import requests

# Define the raw GitHub URL for the new Excel file
excel_formal_url = "https://github.com/emily-escudero/Analitica-Educacion-rural/raw/main/anex-educacionformal.xlsx"

# Define the local path to save the new Excel file
excel_formal_file_path = "anex-educacionformal.xlsx"

# Download the Excel file
print(f"Downloading {excel_formal_url}...")
response = requests.get(excel_formal_url)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

with open(excel_formal_file_path, 'wb') as f:
    f.write(response.content)

print(f"File downloaded to {excel_formal_file_path}")

File downloaded to anex-educacionformal.xlsx


### Preparación de Datos de Sedes Educativas

Ahora voy a cargar los datos de la hoja `Sedes_nivel educativo_zona`. Según tus indicaciones:
*   Los departamentos están en la columna `A` desde la fila `19` hasta la `51`.
*   El número de sedes urbanas va desde la columna `B` hasta la `F` en la fila `17`.
*   El número de sedes rurales va desde la columna `G` hasta la `K` en la fila `17`.

In [16]:
# Load the 'Sedes_nivel educativo_zona' sheet

# 1. Read the headers for urban/rural categories (from row 17, B17:K17)
category_headers = pd.read_excel(
    excel_formal_file_path,
    sheet_name='Sedes_nivel educativo_zona',
    header=None,
    skiprows=16, # Skip rows 0-15, so start reading from row 17 (index 16)
    nrows=1,     # Read only this one row for headers
    usecols='B:K' # Columns B to K
)
raw_category_names = category_headers.iloc[0].tolist()

# Define explicit column names based on their type (Urban/Rural)
# The first 5 raw category names are for Urban, the next 5 are for Rural.
explicit_urban_cols = [f'U_{name}' for name in raw_category_names[:5]] # e.g., U_Preescolar
explicit_rural_cols = [f'R_{name}' for name in raw_category_names[5:]] # e.g., R_Preescolar
all_data_columns_for_numerical_data = explicit_urban_cols + explicit_rural_cols


# 2. Read the department names (from A19 to A51)
departamento_data = pd.read_excel(
    excel_formal_file_path,
    sheet_name='Sedes_nivel educativo_zona',
    header=None,
    skiprows=18, # Skip rows 0-17, so start reading from row 19 (index 18)
    nrows=33,    # Read 33 rows (from A19 to A51)
    usecols='A'  # Only column A for departments
)
departamento_list = departamento_data.iloc[:, 0].tolist()

# 3. Read the actual numerical data block for urban/rural counts (from B19 to K51)
numerical_data = pd.read_excel(
    excel_formal_file_path,
    sheet_name='Sedes_nivel educativo_zona',
    header=None, # No header in this read, we'll assign it manually
    skiprows=18, # Skip rows 0-17, so start reading from row 19 (index 18)
    nrows=33,    # Read 33 rows of data (from row 19 to row 51)
    usecols='B:K' # Columns B to K
)

# Assign the explicitly constructed unique column names
numerical_data.columns = all_data_columns_for_numerical_data

# Assemble the df_instituciones DataFrame
df_instituciones_temp = pd.DataFrame(numerical_data)
df_instituciones_temp.insert(0, 'departamento', departamento_list)

# Fill any potential NaN values in numeric columns with 0
df_instituciones_temp.iloc[:, 1:] = df_instituciones_temp.iloc[:, 1:].fillna(0)



df_instituciones = pd.DataFrame({
    'departamento': df_instituciones_temp['departamento'],
    'total_sedes_urbanas': df_instituciones_temp[explicit_urban_cols].sum(axis=1),
    'total_sedes_rurales': df_instituciones_temp[explicit_rural_cols].sum(axis=1)
})

df_instituciones['total_sedes'] = df_instituciones['total_sedes_urbanas'] + df_instituciones['total_sedes_rurales']

# Display the final DataFrame to confirm the correction
print("Total de Instituciones Urbanas y Rurales por Departamento:")
display(df_instituciones.head())


Total de Instituciones Urbanas y Rurales por Departamento:


,departamento,total_sedes_urbanas,total_sedes_rurales,total_sedes
0,Amazonas,47,161,208
1,Antioquia,4126,9853,13979
2,Arauca,277,846,1123
3,"Archipiélago de San Andrés, Providencia y Sant...",46,35,81
4,Atlántico,3210,271,3481


### Análisis de Datos de Vivienda de TerriData

Ahora, trabajaré con el archivo `TerriData_vivienda.xlsx` para extraer y analizar los indicadores rurales solicitados.

In [34]:
import io, requests, openpyxl, pandas as pd

url = "https://raw.githubusercontent.com/emily-escudero/Analitica-Educacion-rural/main/TerriData_vivienda.xlsx"
indicadores_rurales = [
    "Cobertura de energía eléctrica rural",
    "Cobertura de acueducto rural",
    "Déficit habitacional cualitativo en centros poblados y rural disperso",
    "Déficit habitacional cuantitativo centros poblados y rural disperso",
    "Déficit habitacional en centros poblados y rural disperso",
]

contenido = requests.get(url, timeout=120).content
ws = openpyxl.load_workbook(io.BytesIO(contenido), data_only=True, read_only=True)["Hoja01"]

registros = []
for cod_dep, dep, cod_ent, ent, dim, subcat, indicador, dato, _, anio, *_ in ws.iter_rows(min_row=2, values_only=True):
    if cod_dep is None or dep == "Colombia" or anio not in (2021, 2022, 2023, 2024):
        continue
    cod_dep = str(cod_dep).zfill(2)
    es_departamento = str(cod_ent) == cod_dep + "000" or (cod_dep == "11" and str(cod_ent) == "11001")
    if es_departamento and indicador in indicadores_rurales:
        valor = float(str(dato).replace(".", "").replace(",", ".")) if dato else None
        registros.append((dep, anio, indicador, valor))

df_vivienda_rural = pd.DataFrame(registros, columns=["departamento", "ano", "indicador", "valor"]).pivot(
    index=["departamento", "ano"], columns="indicador", values="valor"
).reset_index()

# Nota: San Andrés y Providencia no tiene dato en los indicadores de déficit habitacional rural
# (no tiene zona "centros poblados y rural disperso" catalogada). Es la única ausencia real.

df_vivienda_rural

indicador,departamento,ano,Cobertura de energía eléctrica rural,Déficit habitacional cualitativo en centros poblados y rural disperso,Déficit habitacional cuantitativo centros poblados y rural disperso,Déficit habitacional en centros poblados y rural disperso
0,Amazonas,2021,26.00,13.98,84.71,98.69
1,Amazonas,2022,26.75,5.60,93.68,99.28
2,Amazonas,2023,41.62,9.50,89.60,99.10
3,Amazonas,2024,27.11,2.70,94.10,96.80
4,Antioquia,2021,98.18,48.00,17.02,65.02
...,...,...,...,...,...,...
127,Vaupés,2024,49.21,2.20,97.80,100.00
128,Vichada,2021,10.83,38.98,54.13,93.11
129,Vichada,2022,13.36,32.94,52.83,85.77
130,Vichada,2023,12.15,21.40,66.40,87.80
